In [1]:
import sys
from os.path import dirname
sys.path.append("../lattice-estimator")

from estimator import *

Logging.set_level(Logging.LEVEL0)


f = 3^5 * 2^3
phi = euler_phi(f)


In [2]:
L1 = 2917 #Q
L2 = 3889 # FS
L3 = 4861 # Q
L4 = 9721 #FS
L5 = 12637 # Q
L6 = 17497 # L
L7 = 19441 #L

tau = 28 # weigth
gamma = 12 * sqrt(3)

In [3]:
# Module-BKZ correction, following Ducas--Engelberts--de Perthuis, "Predicting Module-Lattice Reduction"
# (ePrint 2025/1904).  Condensed from their predictions.py / pred_gain.py, "upper bound" setting.
#
# Idea: module-BKZ over a field K of degree d inserts a whole rank-1 module b*O_K (d vectors) per SVP call.
# How much profile volume those d vectors eat is governed by the density of O_K as a lattice, i.e. by the
# discriminant gap ln(|Delta_K| / d^d).  It is 0 for power-of-two conductors (no gain, only the d-1 penalty),
# and negative whenever an odd prime divides the conductor (flatter slope, i.e. a smaller blocksize suffices).
# An attacker on Q(zeta_c) may reduce over ANY subfield; for c = 2^a 3^b every subfield containing zeta_3 has the
# same per-degree gap, so the cheapest one, Q(zeta_3) with d = 2, is the attacker's best choice.  For Q(zeta_3)
# the paper's skewness and index corrections (their "lower estimate") vanish, so this estimate is exact within
# their model; for other c it is their "upper bound" (the more conservative of their two curves).
from functools import lru_cache
from math import log, lgamma, pi
import numpy as np

@lru_cache(maxsize=None)
def primes_of(c):
    return tuple(p for p in range(2, c + 1) if c % p == 0 and all(p % r for r in range(2, p)))

@lru_cache(maxsize=None)
def cyclo(c):
    # degree phi(c), log|Delta_K| = d ln c - sum_{p|c} d ln p/(p-1)  [Washington, Prop. 2.7], number of roots of unity
    pfs = primes_of(c)
    d = 1
    for p in pfs:
        e = 0; m = c
        while m % p == 0: m //= p; e += 1
        d *= (p - 1) * p ** (e - 1)
    ldisc = d * log(c) - sum(d * log(p) / (p - 1) for p in pfs)
    mu = 2 * c if c % 2 else c
    return d, ldisc, mu

def lgh(n):
    # log Gaussian heuristic of a unit-volume n-dim lattice (their lghZ; Section 3.1)
    return (log(2) - np.euler_gamma - (n / 2) * log(pi) + lgamma(n / 2 + 1)) / n

def slope(c, beta_K):
    # predicted Q-slope of module-BKZ over Q(zeta_c) with K-rank beta_K blocks (Eq. (3)/(4), Heuristic Claim 2)
    #   t1 = module Gaussian heuristic = lgh(d*beta_K) + ln(mu_K/2)/(d*beta_K)       [Section 4.2]
    #   t2 = ln|Delta_K|/(2d) - ln(d)/2   (the discriminant gap; the whole gain lives here)  [Section 4.3]
    #   t3 (skewness), t4 (index) dropped: this is their "upper bound", exact for d = 2   [Sections 4.4, 4.5]
    # c = 1 gives plain BKZ (d = 1, Delta = 1, mu = 2), i.e. slope = -2 lgh(beta)/(beta-1)
    d, ldisc, mu = cyclo(c)
    t = lgh(d * beta_K) + log(mu / 2) / (d * beta_K) + ldisc / (2 * d) - log(d) / 2
    return -(2 / (d * (beta_K - 1))) * t

def beta_eq(beta, c=3):
    # Z-dimension of the SVP oracle module-BKZ over Q(zeta_c) needs to reach the slope of BKZ-beta.
    # Integer search over the K-rank, then linear interpolation (same as their find_beta_eq).
    d = cyclo(c)[0]
    s = slope(1, beta)
    b = max(beta // d, 3)
    sg = 1 if slope(c, b) < s else -1
    while sg * slope(c, b) < sg * s: b += sg
    b_ = b - sg * (s - slope(c, b)) / (slope(c, b - sg) - slope(c, b))
    return d * b_

@lru_cache(maxsize=None)
def coeff_to_canonical(c):
    # NOT from the paper (their Open Question 5).  Module-BKZ must work in the canonical embedding, while the
    # SIS bound is on coefficient vectors.  Gram matrix of the trace form on the power basis of Z[zeta_c] has
    # entries Tr(zeta^(i-j)) = Ramanujan sums; its eigenvalues lam_i tell how the two norms disagree per direction.
    # For a short vector in a random direction, E||coeff||^2 = ||canon||^2 * mean(1/lam)  -> bound scales by
    # sqrt(harmonic mean); the lattice volume scales by sqrt(geometric mean) = |Delta_K|^(1/2d).
    # Returned factor = sqrt(harm/geo) multiplies the length bound; = 1 for power-of-two c, 0.931 for c = 2^a 3^b.
    d = cyclo(c)[0]
    mob = lambda m: 0 if any(m % (p * p) == 0 for p in primes_of(m)) else (-1) ** len(primes_of(m))
    tr = lambda k: mob(c // np.gcd(k, c)) * d // cyclo(c // np.gcd(k, c))[0]
    trv = [tr(k) for k in range(c)]
    lam = np.linalg.eigvalsh(np.array([[trv[(i - j) % c] for j in range(d)] for i in range(d)], dtype=float))
    return float(np.sqrt(1 / np.mean(1 / lam) / np.exp(np.mean(np.log(lam)))))

def sis_mbkz(params, c=3, ring_conductor=None, name=""):
    # 1. run the estimator's usual SIS lattice attack (optionally with the embedding-distorted bound),
    # 2. translate its blocksize to the module-BKZ equivalent over Q(zeta_c), rounded to a multiple of deg,
    # 3. re-cost with the same reduction cost model (RC.MATZOV) at the same lattice dimension d.
    # The unstructured attacker never needs the distortion, so compare mbits against the undistorted BKZ bits.
    f = coeff_to_canonical(ring_conductor) if ring_conductor else 1.0
    cost = SIS.lattice(params.updated(length_bound=params.length_bound * f))
    beta, d = int(cost["beta"]), int(cost["d"])
    dK = cyclo(c)[0]
    be = beta_eq(beta, c)
    be_r = int(round(be / dK)) * dK
    bits, mbits = float(log(cost["rop"], 2)), float(log(RC.MATZOV(be_r, d), 2))
    print(f"{name:14s} BKZ: beta={beta:4d} d={d} {bits:6.1f} bits | mBKZ/Q(z{c}): beta_eq={be:6.1f} {mbits:6.1f} bits | distortion x{f:.3f}")
    return mbits


In [4]:
# observed 
corr0_95 = 2.0
corr0_98 = 3.07
corr0_999 = 3.85

def expected_norm(dim,coo = corr0_999):
    return sqrt(dim * tau * coo)

def account_for_extraction(norm):
    return norm * 8 * gamma

def with_dropped_bits(norm, d, r, coo = corr0_999, height=1):
    return sqrt(norm^2 + height * phi * r * tau * (4^d - 1) / 12 * coo)


In [5]:
# S

params = SIS.Parameters(n=phi, q=L4 * L5, length_bound=account_for_extraction(expected_norm(2^16 * 648, corr0_95)), norm=2)
sis_mbkz(params, ring_conductor=f)
costs = SIS.estimate(params) 

               BKZ: beta= 269 d=1524  106.9 bits | mBKZ/Q(z3): beta_eq= 250.2  101.7 bits | distortion x0.931
lattice  :: rop: ≈2^105.8, red: ≈2^105.8, δ: 1.005255, β: 265, d: 1517, tag: euclidean


In [6]:
params = SIS.Parameters(n=phi, q=L1 * L2 * L3, length_bound=account_for_extraction(with_dropped_bits(expected_norm(2^16 * 648), d=10, r=2^7, coo=corr0_95)), norm=2)
sis_mbkz(params, ring_conductor=f)


               BKZ: beta= 262 d=1740  105.2 bits | mBKZ/Q(z3): beta_eq= 243.4  100.2 bits | distortion x0.931


100.2349167191406

In [7]:
# M

params = SIS.Parameters(n=phi, q= L1 * L2 * L3, length_bound=account_for_extraction(expected_norm(2^18 * 648)), norm=2)
sis_mbkz(params, ring_conductor=f)
costs = SIS.estimate(params) 

               BKZ: beta= 341 d=1901  127.1 bits | mBKZ/Q(z3): beta_eq= 319.3  121.3 bits | distortion x0.931
lattice  :: rop: ≈2^126.0, red: ≈2^126.0, δ: 1.004479, β: 337, d: 1893, tag: euclidean


In [8]:
params = SIS.Parameters(n=phi, q=L1 * L2 * L3, length_bound=account_for_extraction(with_dropped_bits(expected_norm(2^18 * 648), d=9, r=2^8, coo=corr0_95)), norm=2)
sis_mbkz(params, ring_conductor=f)
costs = SIS.estimate(params) 


               BKZ: beta= 276 d=1770  109.1 bits | mBKZ/Q(z3): beta_eq= 256.9  103.6 bits | distortion x0.931
lattice  :: rop: ≈2^108.0, red: ≈2^108.0, δ: 1.005166, β: 272, d: 1763, tag: euclidean


In [9]:
# L

params = SIS.Parameters(n=phi, q= L1 * L2 * L3, length_bound=account_for_extraction(expected_norm(2^20 * 648)), norm=2)
sis_mbkz(params, ring_conductor=f)
costs = SIS.estimate(params) 

               BKZ: beta= 303 d=1826  116.6 bits | mBKZ/Q(z3): beta_eq= 282.8  110.8 bits | distortion x0.931
lattice  :: rop: ≈2^115.5, red: ≈2^115.5, δ: 1.004853, β: 299, d: 1818, tag: euclidean


In [10]:
params = SIS.Parameters(n=phi, q=L1 * L2 * L3, length_bound=account_for_extraction(with_dropped_bits(expected_norm(2^20 * 648), d=8, r=2^9, coo=corr0_95)), norm=2)
sis_mbkz(params, ring_conductor=f)
costs = SIS.estimate(params) 

               BKZ: beta= 281 d=1782  110.5 bits | mBKZ/Q(z3): beta_eq= 261.7  105.2 bits | distortion x0.931
lattice  :: rop: ≈2^109.6, red: ≈2^109.6, δ: 1.005093, β: 278, d: 1775, tag: euclidean


In [11]:
# XL

params = SIS.Parameters(n=phi, q= L1 * L2 * L4, length_bound=account_for_extraction(expected_norm(2^22 * 648)), norm=2)
sis_mbkz(params, ring_conductor=f)
costs = SIS.estimate(params) 

               BKZ: beta= 281 d=1806  110.5 bits | mBKZ/Q(z3): beta_eq= 261.7  105.3 bits | distortion x0.931
lattice  :: rop: ≈2^109.4, red: ≈2^109.4, δ: 1.005105, β: 277, d: 1799, tag: euclidean


In [12]:
params = SIS.Parameters(n=phi, q=L1 * L2 * L3, length_bound=account_for_extraction(with_dropped_bits(expected_norm(2^22 * 648), d=7, r=2^10, coo=corr0_95)), norm=2)
sis_mbkz(params, ring_conductor=f)
costs = SIS.estimate(params) 

               BKZ: beta= 266 d=1749  106.3 bits | mBKZ/Q(z3): beta_eq= 247.3  101.3 bits | distortion x0.931
lattice  :: rop: ≈2^105.2, red: ≈2^105.2, δ: 1.005294, β: 262, d: 1742, tag: euclidean
